# **Exhaustive Certification**

This file has the following part: 

$\S0$ **Setup**: Define global constants, and load packages \
$\S1$ **Construction**: Code to determine the normality and singularity data of hypersurfaces \
$\S2$ **Terminal**: Code to determine whether a point associated to a hypersurface is terminal or not \
$\S3$ **Certification**: Code to perform terminal certification

## $\S0$ Setup

In [17]:
#Constants
alpha_X = -1 #adjunction number of hypersurface

In [19]:
#Packages

#Construction
import math
import numpy as np
import itertools as it

#Terminal
import heapq
import random as rd

#Certification
import pandas as pd
import pickle
import csv
import sqlite3
from collections import defaultdict, Counter

## $\S1$ Construction

Normality

In [ ]:
#normal(degree, point): outputs True if point is normal, else outputs False
def normal(degree, point):
    if not linear_cone(degree, point):
        if well_formed(point):
            if condition_1(degree, point) and condition_2(degree, point) and condition_3(degree, point):
                return True
    return False

#############################################################################################################

#linear_cone(degree, point): outputs True if point is in the linear cone, else outputs False
def linear_cone(degree, point):
    for a in point:
        if a == degree:
            return True
    return False

#wellformedness(degree, point): returns True if the point is well-formed, False otherwise
def well_formed(point): 
    n = len(point) - 1
    for I in it.combinations(range(n + 1), n):
        weights_I = tuple(point[i] for i in I)
        h_I = math.gcd(*weights_I)
        if h_I != 1:
            return False
    return True

#condition_1(degree, point): outputs True if condition 1 is satisfied, else outputs False
def condition_1(degree, point):
    if mu(degree, point) <= 1:
        return False
    return True

#condition_2(degree, point): outputs True if condition 2 is satisfied, else outputs False
def condition_2(degree, point):
    n = len(point) - 1
    for I in it.combinations(range(n + 1), n):
        weights_I = tuple(point[i] for i in I)
        if mu(degree, weights_I) == 0:
            return False
    return True

#condition_3(degree, point): outputs True if condition 3 is satisfied, else outputs False
def condition_3(degree, point):
    n = len(point) - 1
    for I in it.combinations(range(n + 1), n - 1):
        weights_I = tuple(point[i] for i in I)
        h_I = math.gcd(*weights_I)
        if degree % h_I != 0:
            return False
        if rho(degree, point, weights_I) == 0:
            return False
    return True

## $\S2$ Terminal

Singularity Data

In [ ]:
#singularities(degree, point): returns list of singularities if all singularities are terminal, False otherwise
def singularities(degree, point):
    sings = []
    point_strata = strata(point)
    for stratum in point_strata:
        h_I, stratum_weights, stratum_indices, k = stratum
        e = degree % h_I
        mu_I = mu(degree, stratum_weights)
        if mu_I == 1:
            continue
        elif mu_I >= 2:
            dim_qs = k - 1
            qsness = "qs"
            if h_I == 1:
                continue 
            else: #h_I >= 2
                sing_type = "quotient"
                t = tangent_index(degree, point, stratum_weights, stratum_indices)
                stratum_indices_without_t = tuple(i for i in stratum_indices if i != t)
                sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices_without_t[0] if i != t)
                count = counting(degree, point, dim_qs, stratum_weights, qsness)
                sing = (h_I, sing_weights, e, stratum_weights, dim_qs, count, sing_type, qsness)
                sings.append(sing)
        else: #mu_I == 0
            dim_qs = k
            rho_I = rho(degree, point, stratum_weights)
            stratum_indices_complement = tuple(i for i in range(len(point)) if i not in stratum_indices)
            nonqs_existence  = rank_test(degree, point, stratum_weights, stratum_indices)
            if rho_I > k or nonqs_existence == False:
                qsness = "qs"
                if h_I == 1:
                    continue 
                else: #h_I >= 2
                    sing_type = "quotient"
                    t = tangent_index(degree, point, stratum_weights, stratum_indices_complement)
                    sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0] if i != t)
                    count = counting(degree, point, dim_qs, stratum_weights, qsness)
                    sing = (h_I, sing_weights, e, stratum_weights, dim_qs, count, sing_type, qsness)
                    sings.append(sing)
            else: #rho_I<=k and nonqs_existence == True
                if rho_I == 0:
                    qsness = "nonqs"
                    dim_nonqs = k
                    if h_I == 1:
                        sing_type = "hypersurface"
                        sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0])
                        count = counting(degree, point, dim_qs, stratum_weights, qsness)
                        sing = (h_I, sing_weights, e, stratum_weights, dim_nonqs, count, sing_type, qsness)
                        sings.append(sing)
                    else: #h_I >= 2
                        sing_type = "hyperquotient"
                        sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0])
                        count = counting(degree, point, dim_qs, stratum_weights, qsness)
                        sing = (h_I, sing_weights, e, stratum_weights, dim_nonqs, count, sing_type, qsness)
                        sings.append(sing)
                else: #1<=rho_I<= k
                    dim_nonqs = k - rho_I
                    if h_I == 1:
                        qsness= "nonqs"
                        sing_type = "hypersurface"
                        sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0])
                        count = counting(degree, point, dim_qs, stratum_weights, qsness)
                        sing = (h_I, sing_weights, e, stratum_weights, dim_nonqs, count, sing_type, qsness)
                        sings.append(sing)
                    else: #h_I >= 2
                        qsness= "qs"
                        sing_type = "quotient"
                        t = tangent_index(degree, point, stratum_weights, stratum_indices_complement) 
                        sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0] if i != t)
                        count = counting(degree, point, dim_qs, stratum_weights, qsness)
                        sing = (h_I, sing_weights, e, stratum_weights, dim_qs, count, sing_type, qsness)
                        sings.append(sing)
                        qsness= "nonqs"
                        sing_type = "hyperquotient"
                        sing_weights = tuple(point[i] % h_I for i in range(len(point)) if i != stratum_indices[0])
                        count = counting(degree, point, dim_qs, stratum_weights, qsness)
                        sing = (h_I, sing_weights, e, stratum_weights, dim_nonqs, count, sing_type, qsness)
                        sings.append(sing)
    return sings


#############################################################################################################

#strata(point): returns list of strata, where each stratum is given by (h_I, stratum_weights, stratum_indices, k)
#h_I is the gcd of the stratum weights, k is the dimension of the stratum
def strata(point):
    strata = []
    for stratum_len in range(1, len(point) + 1):
        for stratum_indices in it.combinations(range(len(point)), stratum_len):
            stratum_weights = tuple(point[i] for i in stratum_indices)
            h_I = math.gcd(*stratum_weights)
            k = len(stratum_weights) - 1
            stratum = (h_I, stratum_weights, stratum_indices, k)
            strata.append(stratum)
    return tuple(strata)

#monomial_exists(degree, stratum_weights): returns True if there exists a monomial of the given degree that can be formed using the stratum weights, False otherwise
def monomial_exists(degree, stratum_weights):
    def monomial_builder_recursive(total,index):
        if index == len(stratum_weights):
            return total == degree
        for m in range((degree - total) // stratum_weights[index] + 1):
            if monomial_builder_recursive(total + m * stratum_weights[index], index + 1):
                return True
        return False
    return monomial_builder_recursive(0, 0)

#tangent_index(degree, point, stratum_weights, search_indices): returns the index of a tangent variable if it exists, None otherwise
def tangent_index(degree, point, stratum_weights, search_indices):
    for i in search_indices:
        if monomial_exists(degree - point[i], stratum_weights):
            return i
    return None

#mu(degree, stratum_weights): returns the number of monomials of the given degree that can be formed using the stratum weights
def mu(degree, stratum_weights):
    combinations = [0] * (degree + 1) # combinations[i] = number of ways to make i from stratum_weights
    combinations[0] = 1 
    for w in stratum_weights:
        for i in range(w, degree + 1): 
            combinations[i] += combinations[i - w]
        if combinations[degree] >= 2:
            return 2
    return combinations[degree]

#rho(degree, point, stratum_weights): returns the number of tangent variables
def rho(degree, point, stratum_weights):
    rho_count = 0
    for a_t in point:
        if mu(degree - a_t, stratum_weights) >= 1:
            rho_count += 1
    return rho_count

#monomial_finder(degree, stratum_weights, monomial_blacklist): returns a monomial of the given degree that can be formed using the stratum weights and is not in the monomial blacklist, None if no such monomial exists
def monomial_finder(degree, stratum_weights, monomial_blacklist):
    n = len(stratum_weights)
    def monomial_builder_recursive(index, monomial):
        total = sum(monomial[i] * stratum_weights[i] for i in range(n))
        if index == n:
            return monomial if (total == degree) and (monomial not in monomial_blacklist) else None
        for m in range((degree - total) // stratum_weights[index] + 1):
            new_monomial = tuple(monomial[i] if i != index else m for i in range(n))
            result = monomial_builder_recursive(index + 1, new_monomial)
            if result is not None:
                return result
        return None
    return monomial_builder_recursive(0, (0,) * n)

#rank_test(degree, point, stratum_weights, search_indices): returns True if the rank condition is satisfied, False otherwise
def rank_test(degree, point, stratum_weights, stratum_indices): 
    stratum_indices_complement = tuple(i for i in range(len(point)) if i not in stratum_indices)
    J = []
    for i in stratum_indices_complement:
        if monomial_exists(degree - point[i], stratum_weights):
            J.append(i)
    if len(J) == 0:
        return True 
    #If any mu(g_j)=1, then fail
    elif any(mu(degree - point[j], stratum_weights) == 1 for j in J):
        return False
    #Else, mu(g_j)>=2 for all j in J
    else:
        monomials_Y = [[] for _ in range(len(point))] # monomials_Y[i] will hold the list of monomials for g_i
        def grab_new_monomial(J):
            for j in J:
                monomial = monomial_finder(degree - point[j], stratum_weights, monomials_Y[j])
                if monomial is not None:
                    monomials_Y[j].append(monomial)
                    return True
            return False  
        def rank_of_MJ(J):
            MJ = []
            for j in J:
                monomials_gj = monomials_Y[j]
                if len(monomials_gj) < 2:
                    continue  # need at least m0_j and one other monomial to get a difference
                m0 = np.array(monomials_gj[0])
                MJ.extend(np.array(m) - m0 for m in monomials_gj[1:])
            return np.linalg.matrix_rank(np.array(MJ))
        grab_available = True
        for l in range(2, len(J) + 1): #automatically satisfies l=1 as mu(gj)>=2 for all j in J
            for I in it.combinations(J, l):
                while rank_of_MJ(I) < l:
                    grab_available = grab_new_monomial(J)
                    if not grab_available:
                        return False
    return True

#counting(degree, point, dim_sing, stratum_weights, qsness): returns the number of singularities of the given type
def counting (degree, point, dim_sing, stratum_weights, qsness):
    if dim_sing == 0:
        k = len(stratum_weights) - 1
        if k == 0:
            a_0 = stratum_weights[0]
            if degree % a_0 == 0:
                count = 0
            else:
                count = 1
            return count
        elif k == 1:
            a_0, a_1 = stratum_weights
            h_I = math.gcd(a_0, a_1)
            def m_0(degree_2):
                m_0 = None
                for n_1 in range(0, math.floor(int(degree_2 /a_1)) + 1):
                    n_0 = (degree_2 - n_1 * a_1) / a_0
                    if n_0.is_integer():
                        if m_0 is None:
                            m_0 = n_0
                        elif n_0 < m_0:
                            m_0 = n_0
                return int(m_0)
            def m_1(degree_2):
                m_1 = None
                for n_0 in range(0, math.floor(int(degree_2 /a_0)) + 1):
                    n_1 = (degree_2 - n_0 * a_0) / a_1
                    if n_1.is_integer():
                        if m_1 is None:
                            m_1 = n_1
                        elif n_1 < m_1:
                            m_1 = n_1
                return int(m_1)
            if qsness == 'qs':
                if degree % a_0 == 0 and degree % a_1 == 0:
                    count = (degree * h_I) / (a_0 * a_1)
                elif degree % a_0 != 0 and degree % a_1 == 0:
                    count = ((degree - m_1(degree) * a_1) * h_I) / (a_0 * a_1)
                elif degree % a_0 == 0 and degree % a_1 != 0:
                    count = ((degree - m_0(degree) * a_0) * h_I) / (a_0 * a_1)
                else:
                    count = ((degree - m_0(degree) * a_0 - m_1(degree) * a_1) * h_I) / (a_0 * a_1)
            else: #qsness == 'nonqs'
                search_indices = [i for i in range(len(point))]
                t =tangent_index(degree, point, stratum_weights, search_indices)
                a_t = point[t]
                degree_2 = degree - a_t
                if degree_2 % a_0 == 0 and degree_2 % a_1 == 0:
                    count = (degree_2 * h_I) / (a_0 * a_1)
                elif degree_2 % a_0 != 0 and degree_2 % a_1 == 0:
                    count = ((degree_2 - m_1(degree_2) * a_1) * h_I) / (a_0 * a_1)
                elif degree_2 % a_0 == 0 and degree_2 % a_1 != 0:
                    count = ((degree_2 - m_0(degree_2) * a_0) * h_I) / (a_0 * a_1)
                else:
                    count = ((degree_2 - m_0(degree_2) * a_0 - m_1(degree_2) * a_1) * h_I) / (a_0 * a_1)
            if count.is_integer():
                return int(count)
            else:
                print('error in counting function k=1', count, degree, point, stratum_weights, qsness)
                return 'unknown'
        elif k == 2:
            a_0, a_1, a_2 = stratum_weights
            search_indices_1 = [i for i in range(len(point))]
            t1 =tangent_index(degree, point, stratum_weights, search_indices_1)
            search_indices_2 = [i for i in range(len(point)) if i != t1]
            t2 =tangent_index(degree, point, stratum_weights, search_indices_2)
            degree_1 = degree - point[t1]
            degree_2 = degree - point[t2]
            N = (degree_1 * degree_2) / (a_0 * a_1 * a_2)
            if N < 2:
                count = 1
                return count
            else:
                print('unknown case in counting function for k=2', degree, point, stratum_weights, qsness)
            return 'unknown'
        else:
            print('unknown case in counting function for k>2', degree, point, stratum_weights, qsness)
            return 'unknown'
    else:
        return 1

Reid--Shepherd-Barron--Tai

In [ ]:
#RSBT(sing): returns True if the quotient singularity is terminal, False otherwise
def RSBT(sing):
    r = sing[0]
    weights_sing = sing[1]
    for k in range(1, r):
        Delta_k = (sum((k * weight) % r for weight in weights_sing))/r - 1
        if Delta_k <= 0:
            return False
    return True

Face Monomials

In [ ]:
#face_monomials_global(degree, point, patch_weight): returns a list of all face monomials on the global equation
def face_monomials_global_finder(degree, point, patch_weight):
    n = len(point)
    patch_index = point.index(patch_weight)
    B = [patch_weight // math.gcd(a, patch_weight) for a in point]  # proven bound: minimal m[i] <= B[i]
    start = tuple([0] * n)
    heap = [(0, start)]
    pushed = {start}
    frontier = [[] for _ in range(patch_weight)]
    results = []       
    results_full = []
    while heap:
        v, m = heapq.heappop(heap)
        t = v % patch_weight
        dominated_for_search = any(all(f[i] <= m[i] for i in range(n)) for f in frontier[t])
        if v > 0 and v <= degree and (degree - v) % patch_weight == 0:
            if not any(all(g[i] <= m[i] for i in range(n)) for g in results):
                q = m[patch_index] + (degree - v) // patch_weight
                full = list(m)
                full[patch_index] = q
                results.append(m)
                results_full.append(tuple(full))
        if dominated_for_search:
            continue
        frontier[t].append(m)
        for i in range(n):
            if m[i] < B[i] and v + point[i] <= degree:
                nm = list(m)
                nm[i] += 1
                nm = tuple(nm)
                if nm not in pushed:
                    pushed.add(nm)
                    heapq.heappush(heap, (v + point[i], nm))
    return results_full

#face_monomials_patch(degree, point, face_monomials_global, patch_weight): returns a list of all face monomials on the patch equation
def face_monomials_patch_finder(point, face_monomials_global, patch_weight):
    patch_index = point.index(patch_weight)
    results_without_patch_weight = [m[:patch_index] + m[patch_index+1:] for m in face_monomials_global]
    return results_without_patch_weight

#face_monomials_local_1_strata(degree, point, sing, face_monomials_global, patch_weight): returns a list of all face monomials on the local equation
def face_monomials_local_1_strata_finder(degree, point, sing, face_monomials_global, patch_weight):
    stratum_weights = sing[3]
    moving_weight = stratum_weights[1]
    patch_index = point.index(patch_weight)
    moving_index = point.index(moving_weight, patch_index + 1) #maybe give indices in sing output instead of the weights!
    complement_indices = [i for i in range(len(point)) if i != patch_index and i != moving_index]
    complement_monomials = [tuple(m[i] if i in complement_indices else 0 for i in range(len(m))) for m in face_monomials_global]
    complement_monomials = list(set(complement_monomials)) #remove duplicates
    nonqs_dimension = sing[4]
    if nonqs_dimension == 0: #Type II
        face_monomials_local = []
        for m in complement_monomials:
            wt_m = np.dot(m, point)
            mu_m = mu(degree - wt_m, stratum_weights)
            if sum(m) >= 2: #not linear
                face_monomials_local.append(m)
            else: #the unique linear monomial #MAYBE HAVE REPEATS, BUT OKAY?
                ei = tuple(1 if i == moving_index else 0 for i in range(len(point)))
                face_monomials_local.append(tuple(x + y for x, y in zip(m, ei)))
        #covert back to patch monomials
        face_monomials_local = [m[:patch_index] + m[patch_index+1:] for m in face_monomials_local]
        return face_monomials_local, [[]], 'Type II'
    else: #Type III
        face_monomials_local = []
        mu_2_monomials = []
        variable_monomials = [[]]
        for m in complement_monomials:
            wt_m = np.dot(m, point)
            mu_m = mu(degree - wt_m, stratum_weights)
            if mu_m <= 1:
                face_monomials_local.append(m)
            else: #mu_m >= 2
                ei = tuple(1 if i == moving_index else 0 for i in range(len(point)))
                mu_2_monomials.append(m)
                variable_monomials=[mu_2_monomials[:i] + [tuple(x + y for x, y in zip(mu_2_monomials[i], ei))] + mu_2_monomials[i+1:] for i in range(len(mu_2_monomials))]
        #covert back to patch monomials
        face_monomials_local = [m[:patch_index] + m[patch_index+1:] for m in face_monomials_local]
        variable_monomials = [[m[:patch_index] + m[patch_index + 1:] for m in monomials] for monomials in variable_monomials]
        return face_monomials_local, variable_monomials, 'Type III'

#face_monomials_local_2_strata(degree, point, sing, face_monomials_global, patch_weight): returns a list of all sufficient monomials on the local equation for a singularity with 2 strata
def suff_monomials_local_2_strata_finder(degree, point, sing, face_monomials_global, patch_weight):
    stratum_weights = sing[3]
    moving_weight_1 = stratum_weights[1]
    moving_weight_2 = stratum_weights[2]
    patch_index = point.index(patch_weight)
    moving_index_1 = point.index(moving_weight_1, patch_index + 1) #maybe give indices in sing output instead of the weights!
    moving_index_2 = point.index(moving_weight_2, moving_index_1 + 1)
    complement_indices = [i for i in range(len(point)) if i != patch_index and i != moving_index_1 and i != moving_index_2]
    complement_monomials = [tuple(m[i] if i in complement_indices else 0 for i in range(len(m))) for m in face_monomials_global]
    complement_monomials = list(set(complement_monomials)) #remove duplicates
    nonqs_dimension = sing[4]
    strata_dimension = len(stratum_weights) - 1
    if nonqs_dimension < strata_dimension: #Type II
        type_II_or_III = 'Type II'
    else: #Type III
        type_II_or_III = 'Type III'
    if nonqs_dimension == 0: #Type II
        transverse_monomials = []
        for m in complement_monomials:
            wt_m = np.dot(m, point)
            mu_m = mu(degree - wt_m, stratum_weights)
            if sum(m) == 1: #one of the two linear monomials
                transverse_monomials.append(m)
        ei = tuple(1 if i == moving_index_1 else 0 for i in range(len(point)))
        ej = tuple(1 if i == moving_index_2 else 0 for i in range(len(point)))
        es = [ei, ej]
        transverse_monomials = list(set(transverse_monomials)) #remove duplicates
        m1, m2 = transverse_monomials
        variable_monomials = [[tuple(x + y for x, y in zip(m1, e1)), tuple(x + y for x, y in zip(m2, e2))] for e1 in es for e2 in es]
        #covert back to patch monomials
        face_monomials_local = face_monomials_global
        face_monomials_local = [m[:patch_index] + m[patch_index+1:] for m in face_monomials_local] 
        variable_monomials = [[m[:patch_index] + m[patch_index + 1:] for m in monomials] for monomials in variable_monomials]
        return face_monomials_local, variable_monomials, 'Type II'
    else: #nonqs_dimension == 1, Type III
        mu_2_monomials = []
        ei = tuple(1 if i == moving_index_1 else 0 for i in range(len(point)))
        ej = tuple(1 if i == moving_index_2 else 0 for i in range(len(point)))
        es = [ei, ej]
        transverse_monomials = []
        for m in complement_monomials:
            wt_m = np.dot(m, point)
            mu_m = mu(degree - wt_m, stratum_weights)
            if sum(m) >= 2: #not linear
                if mu_m >= 2:
                    mu_2_monomials.append(m)
            else: #unqiue linear monomials
                transverse_monomials.append(m)
        variable_monomials = [[tuple(x + y for x, y in zip(m1, e1)), tuple(x + y for x, y in zip(m2, e2))] for m1 in transverse_monomials for m2 in mu_2_monomials for e1 in es for e2 in es]
        if variable_monomials == []:
            variable_monomials = [[]]
        face_monomials_local = face_monomials_global
        face_monomials_local = [m[:patch_index] + m[patch_index+1:] for m in face_monomials_local]
        variable_monomials = [[m[:patch_index] + m[patch_index + 1:] for m in monomials] for monomials in variable_monomials]
        return face_monomials_local, variable_monomials, 'Type II'


Conditions (A), (B) and (C)

In [ ]:
#condition_A(face_monomials_local): returns True if condition A is satisfied, False otherwise
def condition_A(face_monomials_local):
    n = len(face_monomials_local[0])
    for I in it.combinations(range(n), n - 2):
        I_success = False
        for m in face_monomials_local:
            if any(m[i] >= 1 for i in I) and sum([m[i] for i in range(n) if i not in I]) == 0:
                I_success = True
                break
            else:
                continue
        if not I_success:
            return False
    return True      

#condition_B(sing): returns True if condition B is satisfied, False otherwise
def condition_B(sing):
    r= sing[0]
    sing_weights = sing[1]
    for k in range(1,r):
        sing_weights_k = tuple((sing_weight*k)%r for sing_weight in sing_weights)
        I_k = tuple(i for i, weight in enumerate(sing_weights_k) if weight == 0)
        if len(sing_weights_k) - len(I_k) < 3:
            return False
    return True

#condition_C(face_monomials_local): returns True if condition C is satisfied, False otherwise
def condition_C(face_monomials_local):
    n = len(face_monomials_local[0])
    for I in it.combinations(range(n), n - 3):
        I_success = False
        for m in face_monomials_local:
            if any(m[i] >= 1 for i in I) and sum([m[i] for i in range(n) if i not in I]) <= 1:
                I_success = True
                break
            continue
        if not I_success:
            return False
    return True

Matrix Game

In [ ]:
#Lambda_finder(face_monomials_local): returns a list of all subsets Lambda of {0,1,...,n-1} for which M^0_Lambda needs to be tested for# 
def Lambda_finder(face_monomials_local):
    n = len(face_monomials_local[0])
    Lambdas = []
    for k in range(n):
        for I in it.combinations(range(n), k):
            I_success = False
            for m in face_monomials_local:
                if any(m[i] >= 1 for i in I) and sum([m[i] for i in range(n) if i not in I]) <= 1:
                    I_success = True
                    break
                else:
                    continue
            if not I_success:
                Lambdas.append(list(I))
    return Lambdas

#Delta(k, sing, m, Lambda=None): returns the value of Delta_k for the given k, sing, m and Lambda
def Delta(k, sing, m, Lambda=None):
    r = sing[0]
    sing_weights = sing[1]
    sing_weights_k = tuple((sing_weight*k)%r for sing_weight in sing_weights)
    if k == 0:
        Delta_k = (sum(tuple((1-m[i]) for i in range(len(m)) if i not in Lambda)))-1
    else:
        Delta_k = (sum(tuple((1-m[i])*((sing_weights_k[i])) for i in range(len(m)))))/r-1
    return Delta_k

#matrix_maker(k, point, sing, face_monomials_local, Lambda=None): returns the matrix M^0_Lambda for the given k, point, sing, face_monomials_local and Lambda
def matrix_maker(k, point, sing, face_monomials_local, Lambda=None):
    m_columns = []
    for m in face_monomials_local:
        Delta_k = Delta(k, sing, m, Lambda=Lambda)
        m_column = [Delta_k] + list(np.subtract([1]*len(m),m))
        m_columns.append(m_column)
    Delta_matrix = np.array(m_columns).T
    identity = np.identity(len(point)).astype(int)
    matrix=np.hstack((identity,Delta_matrix))
    return matrix

#matrix_game_player(matrix): plays the matrix game on the given matrix and returns True if the player can win, False otherwise
def matrix_game_player(matrix):
    matrix_copy = matrix.copy()
    #prevents being stuck in a loop if wrong forced move made
    max_forced_moves = 100
    max_attempts = 100
    for _ in range(max_attempts):
        for i in range(max_forced_moves):
            if loss(matrix):
                return False, 'loss' if i == 0 else 'forced_loss'
            if one_monomial(matrix):
                return True, 'one_monomial' if i == 0 else 'forced_one_monomial'
            if two_monomial(matrix):
                return True, 'two_monomial' if i == 0 else 'forced_two_monomial'
            matrix, forced_move_made = forced_move(matrix)
            if not forced_move_made:
                break
        matrix = matrix_copy.copy()
    return None, 'giveup'

#loss(matrix): returns True if the player has lost, False otherwise
def loss(matrix):
    if all(matrix[0,len(matrix):]<=0):
        return True
    else:
        return False

#one_monomial(matrix): returns True if the game can be won with one monomial, False otherwise
def one_monomial(matrix):
    for i in range(len(matrix), len(matrix.T)):
        column = matrix[:,i]
        Delta = column[0]
        if Delta > 0 and all(column[1:]>=0):
            return True
    return False

#two_monomial(matrix): returns True if the game can be won with two monomials, False otherwise
def two_monomial(matrix):
    for i in range(len(matrix), len(matrix.T)):
        for j in range(i + 1, len(matrix.T)):
                column_i = matrix[:,i]
                column_j = matrix[:,j]
                Delta_i = column_i[0]
                Delta_j = column_j[0]
                if all(column_i[1:] >= -1) and all(column_j[1:] >= -1):
                    if any(column_i[1:] < 0) and any(column_j[1:] < 0):
                        if (Delta_i > 0 and Delta_j + math.ceil(Delta_i) > 0) or (Delta_j > 0 and Delta_i + math.ceil(Delta_j) > 0):
                            if all(column_i[1:] + column_j[1:] >= 0):
                                return True
    return False

#forced_move(matrix): makes a forced move on the given matrix if possible and returns the new matrix and a boolean indicating whether a forced move was made
def forced_move(matrix):
    #randomise order of columns to decrease chance of getting stuck in a loop
    indices = list(range(len(matrix), len(matrix.T)))
    rd.shuffle(indices)
    for i in indices:
        column = matrix[:,i]
        Delta = column[0]
        negative_indices = [j for j in range(1, len(column)) if column[j] < 0]
        if Delta > 0 and len(negative_indices) == 1:
            forced_row = matrix[negative_indices[0],:]
            matrix[0,:] = matrix[0,:] + forced_row
            return matrix, True
    return matrix, False

## $\S3$ Certification

Certification

In [ ]:
#terminal_certification(degree, point): returns True if the hypersurface is terminal, False otherwise
def terminal_certification(degree, point):
    data = [[degree], point]
    is_terminal = True
    if normal(degree, point):
        sings = singularities(degree, point)
        for sing in sings:
            stratum_weights = sing[3]
            qsmoothness = sing[-1]
            results_sing = []
            if qsmoothness == 'qs': #quasismooth
                results_sing.append('Type I')
                if not RSBT(sing):
                    data.append([sing, 'fail_RSBT'])
                    return False, data
                else:
                    results_sing.append('RSBT')
            else: #sing[-1] == 'nonqs', so is nonquasismooth
                r = sing[0]
                l = len(sing[3])
                patch_weight = stratum_weights[0]
                face_monomials_global = face_monomials_global_finder(degree, point, patch_weight)
                face_monomials_patch = face_monomials_patch_finder(point, face_monomials_global, patch_weight)
                if l == 1:
                    face_monomials_local = face_monomials_patch
                    #Type III
                    results_sing.append('Type III')
                    results_sing_variable = [[[]]]
                    if not condition_A(face_monomials_local):
                        results_sing_variable[-1].append('fail_condition_A')
                        is_terminal = False
                        continue
                    if not condition_B(sing):
                        results_sing_variable[-1].append('fail_condition_B')
                        is_terminal = False
                        continue
                    for k in range(1,r):
                        matrix = matrix_maker(k, point, sing, face_monomials_local)
                        result, reason = matrix_game_player(matrix)
                        results_sing_variable[-1].append((k, reason))
                        if not result:
                            if is_terminal is not False:
                                is_terminal = result
                            continue
                    Lambdas = Lambda_finder(face_monomials_local)
                    for Lambda in Lambdas:
                        matrix = matrix_maker(0, point, sing, face_monomials_local, Lambda=Lambda)
                        result, reason = matrix_game_player(matrix)
                        results_sing_variable[-1].append((0, Lambda, reason))
                        if not result:
                            if is_terminal is not False:
                                is_terminal = result
                            continue
                    results_sing.append(results_sing_variable)
                elif l == 2:
                    face_monomials_local, variable_monomials, type_II_or_III = face_monomials_local_1_strata_finder(degree, point, sing, face_monomials_global, patch_weight)
                    if type_II_or_III == 'Type II': #Type II
                        results_sing.append('Type II')
                        results_sing_variable = [[[]]]
                        if not condition_A(face_monomials_local):
                            results_sing_variable[-1].append('fail_condition_A')
                            is_terminal = False
                            continue
                        if not condition_B(sing):
                            results_sing_variable[-1].append('fail_condition_B')
                            is_terminal = False
                            continue
                        if not condition_C(face_monomials_local):
                            results_sing_variable[-1].append('fail_condition_C')
                            is_terminal = False
                            continue
                        for k in range(1,r):
                            matrix = matrix_maker(k, point, sing, face_monomials_local)
                            result, reason = matrix_game_player(matrix)
                            results_sing_variable[-1].append((k, reason))
                            if not result:
                                if is_terminal is not False:
                                    is_terminal = result
                                continue
                        results_sing.append(results_sing_variable)
                    else: #dim_nonqs = 1, so Type III
                        results_sing.append('Type III')
                        results_sing_variable = []
                        for monomials in variable_monomials:
                            results_sing_variable.append([monomials])
                            face_monomials_local_variable = face_monomials_local + monomials
                            if not condition_A(face_monomials_local_variable):
                                results_sing_variable[-1].append('fail_condition_A')
                                is_terminal = False
                                continue
                            if not condition_B(sing):
                                results_sing_variable[-1].append('fail_condition_B')
                                is_terminal = False
                                continue
                            for k in range(1,r):
                                matrix = matrix_maker(k, point, sing, face_monomials_local)
                                result, reason = matrix_game_player(matrix)
                                results_sing_variable[-1].append((k, reason))
                                if not result:
                                    if is_terminal is not False:
                                        is_terminal = result
                                    continue
                            Lambdas = Lambda_finder(face_monomials_local_variable)
                            for Lambda in Lambdas:
                                matrix = matrix_maker(0, point, sing, face_monomials_local_variable, Lambda=Lambda)
                                result, reason = matrix_game_player(matrix)
                                results_sing_variable[-1].append((0, Lambda, reason))
                                if not result:
                                    if is_terminal is not False:
                                        is_terminal = result
                                    continue
                        results_sing.append(results_sing_variable)
                else:
                    suff_monomials_local, variable_monomials, type_II_or_III = suff_monomials_local_2_strata_finder(degree, point, sing, face_monomials_global, patch_weight)
                    if type_II_or_III == 'Type II': #Type II
                        results_sing.append('Type II')
                        results_sing_variable = []
                        for monomials in variable_monomials:
                            results_sing_variable.append([monomials])
                            face_monomials_variable_suff = suff_monomials_local + monomials
                            if not condition_A(face_monomials_variable_suff):
                                results_sing_variable[-1].append('uncertain_condition_A')
                                is_terminal = None
                                continue
                            if not condition_B(sing):
                                results_sing_variable[-1].append('fail_condition_B')
                                is_terminal = False
                                continue
                            if not condition_C(face_monomials_variable_suff):
                                results_sing_variable[-1].append('uncertain_condition_C')
                                is_terminal = None
                                continue
                            for k in range(1,r):
                                matrix = matrix_maker(k, point, sing, face_monomials_variable_suff)
                                result, reason = matrix_game_player(matrix)
                                results_sing_variable[-1].append((k, reason))
                                if not result:
                                    is_terminal = None
                        results_sing.append(results_sing_variable)
                    else: # Type III
                        results_sing_variable = [[[]]]
                        results_sing_variable[-1].append('Type III with k>1')
                        results_sing.append(results_sing_variable)
                        is_terminal = None
                        #for our case this automatically fails.
            data.append([sing, results_sing])
        return is_terminal, data
    else:
        data.append(['fail_normal'])
        print('fail normal', degree, point)
        is_terminal = False
        return is_terminal, data

Load approximated data

In [ ]:
points = [tuple(map(int, row)) for row in pd.read_csv('../data/approximation/approximation_points.csv', header=None).values]

Run Certification

In [ ]:
counter = 0
terminal_counter = 0
non_terminal_counter = 0
unknown_terminal_counter = 0
certification_raw = []

for point in points:
    counter += 1
    degree = sum(point) + alpha_X
    is_terminal, data  = terminal_certification(degree, point)
    certification_raw.append((is_terminal, data))
    if is_terminal:
        terminal_counter += 1
    elif is_terminal is False:
        non_terminal_counter += 1
    else:
        unknown_terminal_counter += 1
    if counter % 1000 == 0:
        print(f"Processed {counter} points. Terminal: {terminal_counter}, Non-terminal: {non_terminal_counter}, Unknown: {unknown_terminal_counter}")

Save raw data

In [ ]:
os.makedirs("../data", exist_ok=True)

with open("../data/certification/certification_raw.pkl", "wb") as f:
    pickle.dump(certification_raw, f)

Load raw data

In [20]:
import pickle

with open("../data/certification/certification_raw.pkl", "rb") as f:
    certification_raw = pickle.load(f)

Print and study any unconfirmed examples

In [21]:
for is_terminal, data in certification_raw:
    if is_terminal is None:
        print("Unknown terminal status for hypersurface:", data[0:2])
        print("Details:", data)
        print("--------------------------------------------------")

These unconfirmed candidates are then studied and verified. For examples with d ≤ 200 in the exhaustive search, we confirmed that each is terminal by playing the matrix game by hand, introducing the 'complex_forced_move' and 'three_monomial' strategies where needed. The following code does this by hand for such examples. 

In [22]:
complex_forced_one_monomial_cases = [(1, 3, 5, 16, 20, 22), (1, 3, 16, 21, 36, 38), (1, 18, 19, 29, 32, 49), (1, 3, 13, 43, 57, 58), (1, 3, 28, 32, 66, 69)]
complex_forced_two_monomial_cases = [(3, 4, 7, 9, 11, 25), (2, 10, 13, 14, 19, 27)]
forced_three_monomial_cases = [(3, 4, 7, 10, 17, 23), (3, 4, 7, 10, 17, 40), (3, 4, 7, 10, 23, 34), (4, 6, 7, 10, 17, 37), (5, 9, 12, 13, 22, 30), (3, 4, 6, 11, 34, 45), (5, 8, 9, 13, 22, 56), (3, 4, 7, 10, 34, 57)]

case_mappings = [(complex_forced_one_monomial_cases, 'complex_forced_one_monomial'), (complex_forced_two_monomial_cases, 'complex_forced_two_monomial'), (forced_three_monomial_cases, 'forced_three_monomial')]

for is_terminal, data in certification_raw:
    for case_collection, new_reason in case_mappings:
        if data[1] in case_collection:
            certification_raw[certification_raw.index((is_terminal, data))] = (True, data)
            sings = data[2:]
            for sing, results_sing in sings:
                if results_sing[0] == 'Type III':
                    for result in results_sing[1]:
                        for k_reason in result[1:]:
                            k = k_reason[0]
                            reason = k_reason[2] if k == 0 else k_reason[1]
                            if reason == 'giveup':
                                new_k_reason = (*k_reason[0:-1], new_reason)
                                result[result.index(k_reason)] = new_k_reason
            

Break down data into different simplifications

In [23]:
#long
certification_long = []
for is_terminal, data in certification_raw:
    if is_terminal is True:
        certification_long.append(data)

#medium
certification_medium = []
for is_terminal, data in certification_raw:
    if is_terminal is True:
        degree = data[0][0]
        point = data[1]
        sings_data = data[2:]

        # Higher-dimensional singularities
        higher_dim_singularities = [(sing, results_sing[0]) for sing, results_sing in sings_data if sing[4] >= 1]

        sings_medium = []
        for sing, results_sing in sings_data:
            r, weights_sing, e, dim_sing, count_sing = sing[0], sing[1], sing[2], sing[4], sing[5]
            type_sing = results_sing[0]

            # Skip dim-0 entries only when all conditions are satisfied
            if dim_sing == 0:
                if any(
                    type_sing == higher_type
                    and int(r) == int(higher[0])
                    and Counter(weights_sing) == Counter(higher[1])
                    and not (Counter(sing[3]) - Counter(higher[3]))
                    for higher, higher_type in higher_dim_singularities
                ):
                    continue

            sings_medium.append((r, weights_sing, e, dim_sing, count_sing, type_sing))

        certification_medium.append([[degree], point, sings_medium])


#short
certification_short = []
for is_terminal, data in certification_raw:
    degree = data[0][0]
    point = data[1]
    data_short = [[degree], point]
    certification_short.append(data_short)


Save Certified Data

In [24]:
with open("../data/certification/certification_long.pkl", "wb") as f:
    pickle.dump(certification_long, f)

with open("../data/certification/certification_medium.pkl", "wb") as f:
    pickle.dump(certification_medium, f)

with open("../data/certification/certification_short.pkl", "wb") as f:
    pickle.dump(certification_short, f)

with open("../data/certification/certification_short.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(point for degree, point in certification_short)

Load Certified Data

In [ ]:
with open("../data/certification/certification_long.pkl", "rb") as f:
    certification_long = pickle.load(f)

with open("../data/certification/certification_medium.pkl", "rb") as f:
    certification_medium = pickle.load(f)

with open("../data/certification/certification_short.pkl", "rb") as f:
    certification_short = pickle.load(f)

Save Data as SQL

In [ ]:
import os
import math
import sqlite3
from fractions import Fraction

import pandas as pd

# Requires certification_medium to be loaded in the current session.

# --------------------------------------------------
# 1. Settings and quasismooth reference data
# --------------------------------------------------
MAX_DEGREE = 200
output_dir = "../data/certification"
os.makedirs(output_dir, exist_ok=True)

table_name = "terminal_fano_4fold_index1_200"
sql_path = os.path.join(output_dir, f"{table_name}.sql")

# First column is degree; the remaining six columns are weights.
qs_data = pd.read_csv("../data/quasismooth/quasismooth_terminal.csv", header=None).values
qs_set = {tuple(int(w) for w in row[1:]) for row in qs_data}
assert all(len(point) == 6 for point in qs_set)

# Known nonquasismooth exception erroneously included in the reference CSV.
# This family has d = 316, so it does not belong to the d <= 200 export.
non_qs_exception = (1, 2, 15, 63, 79, 157)
if non_qs_exception in qs_set:
    qs_set.remove(non_qs_exception)
    print(f"Excluded known non-QS family X_{sum(non_qs_exception)-1}: {non_qs_exception}")

qs_set_200 = {point for point in qs_set if sum(point) - 1 <= MAX_DEGREE}
print(f"Reference quasismooth families with d <= {MAX_DEGREE}: {len(qs_set_200):,}")

# --------------------------------------------------
# 2. Helper functions and basket columns
# --------------------------------------------------
def compute_degree(degree, point):
    """Exact rational number d/(a1*...*a6)."""
    return Fraction(int(degree), math.prod(int(a) for a in point))


def format_basket(basket, n_vars):
    """Format singularities as in the sporadic certification database."""
    parts = []
    for sing in basket:
        w = ",".join(str(x) for x in sorted(int(x) for x in sing["weights"]))
        if len(sing["weights"]) == n_vars - 1:
            text = f"1/{sing['r']}({w};{sing['e']})"
        else:
            text = f"1/{sing['r']}({w})"
        parts.extend([text] * int(sing["count"]))
    return ",".join(parts)


COLUMNS = {
    ("Type I", "point"): "Type_I_Point",
    ("Type II", "point"): "Type_II_Point",
    ("Type III", "point"): "Type_III_Point",
    ("Type I", "curve"): "Type_I_Curve",
    ("Type II", "curve"): "Type_II_Curve",
    ("Type III", "curve"): "Type_III_Curve",
}
DIM_NAMES = {0: "point", 1: "curve"}

# --------------------------------------------------
# 3. Create database in memory (SQL-only export)
# --------------------------------------------------
con = sqlite3.connect(":memory:")

con.execute(f"""
CREATE TABLE {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    Equation_degree INTEGER NOT NULL CHECK (Equation_degree <= {MAX_DEGREE}),
    a1 INTEGER NOT NULL,
    a2 INTEGER NOT NULL,
    a3 INTEGER NOT NULL,
    a4 INTEGER NOT NULL,
    a5 INTEGER NOT NULL,
    a6 INTEGER NOT NULL,
    CanonicalClass INTEGER NOT NULL,
    Degree TEXT NOT NULL,
    Degree_dec REAL NOT NULL,
    Type_I_Point TEXT,
    Type_II_Point TEXT,
    Type_III_Point TEXT,
    Type_I_Curve TEXT,
    Type_II_Curve TEXT,
    Type_III_Curve TEXT,
    Point_Basket_size INTEGER,
    Curve_Basket_size INTEGER,
    IsSmooth INTEGER CHECK (IsSmooth IN (0, 1)),
    IsQsmooth INTEGER NOT NULL CHECK (IsQsmooth IN (0, 1)),
    IsTerminal INTEGER NOT NULL CHECK (IsTerminal IN (0, 1)),
    IsCanonical INTEGER NOT NULL CHECK (IsCanonical IN (0, 1)),
    UNIQUE (a1, a2, a3, a4, a5, a6)
)
""")

# --------------------------------------------------
# 4. Prepare INSERT statement
# --------------------------------------------------
insert_columns = [
    "Equation_degree", "a1", "a2", "a3", "a4", "a5", "a6",
    "CanonicalClass", "Degree", "Degree_dec",
    "Type_I_Point", "Type_II_Point", "Type_III_Point",
    "Type_I_Curve", "Type_II_Curve", "Type_III_Curve",
    "Point_Basket_size", "Curve_Basket_size",
    "IsSmooth", "IsQsmooth", "IsTerminal", "IsCanonical",
]
insert_sql = f"INSERT INTO {table_name} ({', '.join(insert_columns)}) VALUES ({', '.join('?' for _ in insert_columns)})"

# --------------------------------------------------
# 5. Convert records into SQL rows
# --------------------------------------------------
def make_row(degree, point, sings_medium):
    point = tuple(int(w) for w in point)
    degree = int(degree)

    if len(point) != 6:
        raise ValueError(f"Expected six weights: {point}")
    if degree != sum(point) - 1:
        raise ValueError(f"Incorrect index-one degree: {degree}, {point}")
    if degree > MAX_DEGREE:
        raise ValueError(f"Degree exceeds {MAX_DEGREE}: {degree}, {point}")

    frac = compute_degree(degree, point)

    if sings_medium is None:
        # Quasismooth reference family lacking a medium certification record.
        if point not in qs_set_200:
            raise ValueError(f"Missing singularity information for {point}")
        is_qsmooth = 1
        baskets = {col: None for col in COLUMNS.values()}
        n_points = n_curves = is_smooth = None
    else:
        types = {str(s[5]).strip() for s in sings_medium}
        unexpected_types = types - {"Type I", "Type II", "Type III"}
        if unexpected_types:
            raise ValueError(f"Unexpected singularity types for {point}: {unexpected_types}")

        is_qsmooth = int(not (types & {"Type II", "Type III"}))
        baskets = {col: [] for col in COLUMNS.values()}
        n_points = n_curves = 0

        for r, weights_sing, e, dim_sing, count_sing, type_sing in sings_medium:
            dim_name = DIM_NAMES.get(int(dim_sing))
            col = COLUMNS.get((str(type_sing).strip(), dim_name))
            if col is None:
                raise ValueError(f"Unknown singularity type/dimension for {point}: {type_sing!r}, {dim_sing!r}")
            count_sing = int(count_sing)
            if count_sing < 0:
                raise ValueError(f"Negative singularity count for {point}: {count_sing}")

            sing = {"r": int(r), "weights": tuple(int(w) for w in weights_sing), "e": int(e), "count": count_sing}
            baskets[col].append(sing)
            if dim_name == "point":
                n_points += count_sing
            else:
                n_curves += count_sing

        is_smooth = int(n_points == 0 and n_curves == 0)
        baskets = {col: format_basket(basket, len(point)) for col, basket in baskets.items()}

    values = {
        "Equation_degree": degree,
        "a1": point[0], "a2": point[1], "a3": point[2],
        "a4": point[3], "a5": point[4], "a6": point[5],
        "CanonicalClass": degree - sum(point),
        "Degree": str(frac),
        "Degree_dec": float(frac),
        "Type_I_Point": baskets["Type_I_Point"],
        "Type_II_Point": baskets["Type_II_Point"],
        "Type_III_Point": baskets["Type_III_Point"],
        "Type_I_Curve": baskets["Type_I_Curve"],
        "Type_II_Curve": baskets["Type_II_Curve"],
        "Type_III_Curve": baskets["Type_III_Curve"],
        "Point_Basket_size": n_points,
        "Curve_Basket_size": n_curves,
        "IsSmooth": is_smooth,
        "IsQsmooth": is_qsmooth,
        "IsTerminal": 1,
        "IsCanonical": 1,
    }
    return tuple(values[col] for col in insert_columns)

# --------------------------------------------------
# 6. Insert exhaustive certification families
# --------------------------------------------------
included_points = set()
for record in certification_medium:
    degree_list, point, sings_medium, *rest = record
    point = tuple(int(w) for w in point)
    degree = int(degree_list[0])

    if degree > MAX_DEGREE:
        continue
    if point in included_points:
        raise ValueError(f"Duplicate exhaustive family: {point}")

    con.execute(insert_sql, make_row(degree, point, sings_medium))
    included_points.add(point)

print(f"Exhaustive certification families with d <= {MAX_DEGREE}: {len(included_points):,}")

# --------------------------------------------------
# 7. Add missing quasismooth reference families
# --------------------------------------------------
missing_qs = qs_set_200 - included_points
for point in sorted(missing_qs):
    con.execute(insert_sql, make_row(sum(point) - 1, point, None))
print(f"Additional quasismooth reference families: {len(missing_qs):,}")

# --------------------------------------------------
# 8. Create indexes and verify counts
# --------------------------------------------------
con.execute(f"CREATE INDEX idx_exhaustive_degree ON {table_name}(Equation_degree)")
con.execute(f"CREATE INDEX idx_exhaustive_qsmooth ON {table_name}(IsQsmooth)")
con.execute(f"CREATE INDEX idx_exhaustive_degree_dec ON {table_name}(Degree_dec)")
con.commit()

stats = con.execute(f"""
SELECT COUNT(*),
       SUM(CASE WHEN IsQsmooth = 1 THEN 1 ELSE 0 END),
       SUM(CASE WHEN IsQsmooth = 0 THEN 1 ELSE 0 END),
       MIN(Equation_degree), MAX(Equation_degree)
FROM {table_name}
""").fetchone()

total, n_qs, n_nonqs, min_degree, max_degree = stats
assert total == len(included_points | qs_set_200)
assert total == n_qs + n_nonqs

print("\nDatabase summary")
print("------------------------------")
print(f"Total families:          {total:,}")
print(f"Quasismooth:             {n_qs:,}")
print(f"Nonquasismooth:          {n_nonqs:,}")
print(f"Minimum equation degree: {min_degree}")
print(f"Maximum equation degree: {max_degree}")

# --------------------------------------------------
# 9. Compare quasismooth classification to reference
# --------------------------------------------------
classified_qs = {tuple(row) for row in con.execute(f"SELECT a1, a2, a3, a4, a5, a6 FROM {table_name} WHERE IsQsmooth = 1")}
unexpected_qs = classified_qs - qs_set_200
missing_qs_classification = qs_set_200 - classified_qs

print("\nQuasismoothness verification")
print("------------------------------")
print(f"QS absent from reference: {len(unexpected_qs):,}")
print(f"Reference QS classified non-QS: {len(missing_qs_classification):,}")

if unexpected_qs or missing_qs_classification:
    print("WARNING: QS discrepancies remain; SQL export will proceed.")
    for point in sorted(unexpected_qs)[:10]:
        print(f"  Unexpected QS: X_{sum(point)-1}, {point}")
    for point in sorted(missing_qs_classification)[:10]:
        print(f"  Reference QS classified non-QS: X_{sum(point)-1}, {point}")

# --------------------------------------------------
# 10. Export SQL dump
# --------------------------------------------------
with open(sql_path, "w", encoding="utf-8") as f:
    for statement in con.iterdump():
        f.write(statement + "\n")

print(f"\nSaved SQL dump: {sql_path}")
con.close()
